In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from pathlib import Path

user_secrets = UserSecretsClient()

BASE_URL = "https://neglegere.com"
EMAIL = user_secrets.get_secret("email")
PASSWORD = user_secrets.get_secret("password")
SAVE_DIR = Path("/kaggle/working/reference_audio")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_TOKEN"] = user_secrets.get_secret("hf_token")

R2_ACCOUNT_ID = user_secrets.get_secret("r2_AccountID")
R2_ACCESS_KEY = user_secrets.get_secret("r2_accesskey")
R2_SECRET_KEY = user_secrets.get_secret("r2_secretkey")
R2_BUCKET_NAME = user_secrets.get_secret("r2_bucket_name")
R2_ENDPOINT = f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com"

WEBHOOK_URL = user_secrets.get_secret("webhook_url")


def notify_slack(text: str):
    import requests
    try:
        requests.post(WEBHOOK_URL, json={"text": text}, timeout=15)
    except Exception as e:
        print("Slack通知に失敗:", e)

In [ ]:
# torch/soundfile/transformers/accelerate は Kaggle に同梱済みのため入れない
!pip install -q qwen-tts librosa scipy

import os
import torch
import soundfile as sf
from pathlib import Path
from qwen_tts import Qwen3TTSModel


import signal


class GenerationTimeout(Exception):
    pass


def generate_qwen_voice(
    text: str,
    ref_audio_path: str,
    ref_text: str,
    output_path: str = "/kaggle/working/qwen_generated.wav",
    model_name: str = "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
):
    """
    Qwen3-TTSを使用してボイスクローン音声を生成するメソッド。

    Args:
        text (str): 生成する音声のテキスト
        ref_audio_path (str): 参照音声（WAVファイル）のパス
        ref_text (str): 参照音声の読み上げ内容
        output_path (str): 生成された音声の保存先
        model_name (str): 使用するQwenモデル

    Returns:
        str: 生成された音声ファイルのパス
    """

    # デバイス判定
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # モデルの読み込み
    model = Qwen3TTSModel.from_pretrained(
        model_name,
        device_map="cuda:0" if device == "cuda" else "cpu",
        dtype=torch.float16 if device == "cuda" else torch.float32,
    )

    print(f"✅ Model loaded on {device}")

    # 音声生成（10分でタイムアウトして中断）
    def _on_generate_timeout(signum, frame):
        raise GenerationTimeout("generate_voice_clone が10分を超えたため中断しました")

    old_handler = signal.signal(signal.SIGALRM, _on_generate_timeout)
    signal.alarm(600)
    try:
        wavs, sr = model.generate_voice_clone(
            text=text,
            language="Japanese",
            ref_audio=ref_audio_path,
            ref_text=ref_text,
        )
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

    # 保存先のディレクトリを作成
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # WAVファイルとして保存
    sf.write(str(output_path), wavs[0], sr)

    print(f"✅ Generated voice saved: {output_path}")

    return str(output_path)

In [ ]:
import requests
from pathlib import Path

def login() -> str:
    url = f"{BASE_URL}/api/auth/login"
    headers = {
        "accept": "application/json",
        "Content-Type": "application/json",
    }
    payload = {
        "email": EMAIL,
        "password": PASSWORD,
    }

    res = requests.post(url, headers=headers, json=payload, timeout=30)
    res.raise_for_status()

    data = res.json()

    token = data.get("token")

    return token

def get_pending_items(token: str) -> list[dict]:
    url = f"{BASE_URL}/api/voice_clones/pending"
    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    res = requests.get(url, headers=headers, timeout=30)
    res.raise_for_status()

    data = res.json()

    items = data.get("items", [])

    return items

def download_reference_audio(token: str, voice_clone_id: str, file_name: str | None = None) -> Path:
    url = f"{BASE_URL}/api/reference_audio/{voice_clone_id}/file"
    headers = {
        "accept": "audio/wav",
        "Authorization": f"Bearer {token}",
    }

    res = requests.post(url, headers=headers, data="", stream=True, timeout=60)
    res.raise_for_status()

    output_name = Path(file_name).name

    output_path = SAVE_DIR / output_name

    with open(output_path, "wb") as f:
        for chunk in res.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    return output_path

def upload_completed_voice_clone(
    token: str,
    voice_clone_id: str,
    audio_file_path,
):
    audio_file_path = Path(audio_file_path)

    url = f"{BASE_URL}/api/voice_clones/{voice_clone_id}/complete"

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    with open(audio_file_path, "rb") as f:
        files = {
            "file": (audio_file_path.name, f, "audio/wav")
        }

        res = requests.post(
            url,
            headers=headers,
            files=files,
            timeout=60,
        )

    res.raise_for_status()

    data = res.json()

    if not data.get("ok"):
        raise Exception(f"APIエラー: {data}")

    return data


def fail_voice_clone(token: str, voice_clone_id: str):
    url = f"{BASE_URL}/api/voice_clones/{voice_clone_id}/fail"

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    res = requests.post(url, headers=headers, timeout=30)
    res.raise_for_status()

    return res.json()


In [ ]:
import boto3
import json

def fetch_pending_voice_clones(key: str = "pending.json"):
    s3 = boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )

    res = s3.get_object(
        Bucket=R2_BUCKET_NAME,
        Key=key,
    )

    return json.loads(res["Body"].read().decode("utf-8"))

print(fetch_pending_voice_clones())

In [ ]:
import shutil
import time

def main():

    i = 0
    items = fetch_pending_voice_clones()
    
    if not items:
        print("pending は 0 件です")
        return

    for item in items["items"]:
        Path("/kaggle/working/reference_audio").mkdir(parents=True, exist_ok=True)
        Path("/kaggle/working/clone").mkdir(parents=True, exist_ok=True)
    
        voice_clone_id = item["id"]
        referenceAudioPath = item["referenceAudioPath"]

        print(f"downloading id={voice_clone_id} ...")
        token = login()
        output_path = download_reference_audio(
            token, 
            voice_clone_id, 
            referenceAudioPath
        )
        print(f"saved: {output_path}")

        # 音声生成（10分超はスキップ）
        try:
            output_file = generate_qwen_voice(
                text=item["desiredText"],
                ref_audio_path=str(output_path),
                ref_text=item["recordedText"],
                output_path=Path("/kaggle/working/clone/") / Path(referenceAudioPath).name,
            )
        except GenerationTimeout as e:
            print(f"skip id={voice_clone_id}: {e}")
            notify_slack(
                f":warning: voice-clone-task-by-T4: id={voice_clone_id} が10分を超えたためスキップしました"
            )
            try:
                fail_voice_clone(login(), voice_clone_id)
            except Exception as fe:
                print(f"fail 通知に失敗: {fe}")
            shutil.rmtree("/kaggle/working/reference_audio", ignore_errors=True)
            shutil.rmtree("/kaggle/working/clone", ignore_errors=True)
            continue

        upload_completed_voice_clone(
            token,
            voice_clone_id,
            output_file
        )
        
        print("生成されたファイル:", output_file)
        
        shutil.rmtree("/kaggle/working/reference_audio")
        shutil.rmtree("/kaggle/working/clone")

    print("\n完了")

import traceback

try:
    main()
except Exception:
    tb = traceback.format_exc()
    print(tb)
    notify_slack(
        f":rotating_light: voice-clone-task-by-T4 が失敗しました\n```\n{tb[-3000:]}\n```"
    )
    raise
